
# Experiment 4 — Boundary-Specific Evaluation

This notebook is designed as a **run-all experimental harness**. Edit only the **USER CONFIG** and **MODEL FACTORY** cells before running.

## Required dataset manifest

Prepare a CSV with one row per target axial slice and these columns:

- `patient_id`
- `split` (`train`, `val`, or `test`)
- `image_path` — target slice (`.npy`, `.png`, `.jpg`, `.tif`)
- `mask_path` — binary reference mask
- `prev1_path`, `next1_path` — adjacent slices for 3-slice 2.5D
- optional `prev2_path`, `next2_path` — for 5-slice experiment
- optional `pixel_spacing_x`, `pixel_spacing_y`
- optional `slice_index`

**Important:** all train/validation/test splitting must remain patient-level. Do not allow slices from one patient to appear in different splits.

## Model interface

The notebook expects a function:

`build_model(variant: dict) -> torch.nn.Module`

The model must accept tensors shaped `[B, C, H, W]` and return logits shaped `[B, 1, H, W]`.

Replace the placeholder model factory with your actual CT-SE(2)/Mod-SE(2) implementation. The experiment harness, training, metrics, CSV export, and plots then run automatically.


Compares Dice-only, Focal+Dice, Dice+Boundary, and the full Focal+Dice+Boundary objective using Dice, IoU, HD95, ASSD, and 2-mm Surface Dice.

In [ ]:

import os, random, math, time, copy, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from scipy.ndimage import distance_transform_edt, binary_erosion, label as cc_label
from scipy.optimize import linear_sum_assignment

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)


In [ ]:

# ========================= USER CONFIG =========================
MANIFEST_CSV = "/path/to/manifest.csv"
OUTPUT_DIR = "./outputs"
SEEDS = [42, 123, 2026]

EPOCHS = 150
BATCH_SIZE = 8
LR = 1e-4
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 4
THRESHOLD = 0.5
IMAGE_SIZE = (256, 256)

# Set False to test the notebook pipeline quickly.
FULL_TRAINING = True

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:

def load_array(path):
    path = str(path)
    ext = Path(path).suffix.lower()
    if ext == ".npy":
        x = np.load(path)
    else:
        x = np.array(Image.open(path).convert("F"))
    return x.astype(np.float32)

def resize_array(x, size, is_mask=False):
    im = Image.fromarray(x)
    mode = Image.Resampling.NEAREST if is_mask else Image.Resampling.BILINEAR
    return np.array(im.resize((size[1], size[0]), mode)).astype(np.float32)

class CTSliceDataset(Dataset):
    def __init__(self, df, context=3, augment=False):
        self.df = df.reset_index(drop=True)
        self.context = context
        self.augment = augment

    def __len__(self): return len(self.df)

    def _paths(self, r):
        if self.context == 1:
            return [r.image_path]
        if self.context == 3:
            return [r.prev1_path, r.image_path, r.next1_path]
        if self.context == 5:
            return [r.prev2_path, r.prev1_path, r.image_path, r.next1_path, r.next2_path]
        raise ValueError("context must be 1, 3, or 5")

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        chans = []
        for p in self._paths(r):
            x = resize_array(load_array(p), IMAGE_SIZE, False)
            chans.append(x)
        x = np.stack(chans, 0)
        # volume/slice-wise substitute normalization; replace with exact manuscript preprocessing if needed.
        x = (x - x.mean()) / (x.std() + 1e-6)

        y = resize_array(load_array(r.mask_path), IMAGE_SIZE, True)
        y = (y > 0).astype(np.float32)[None, ...]

        if self.augment and random.random() < 0.5:
            x = x[:, :, ::-1].copy()
            y = y[:, :, ::-1].copy()

        meta = {
            "patient_id": str(r.patient_id),
            "slice_index": int(r.slice_index) if "slice_index" in self.df.columns and pd.notna(r.slice_index) else idx,
            "spacing_x": float(r.pixel_spacing_x) if "pixel_spacing_x" in self.df.columns and pd.notna(r.pixel_spacing_x) else 1.0,
            "spacing_y": float(r.pixel_spacing_y) if "pixel_spacing_y" in self.df.columns and pd.notna(r.pixel_spacing_y) else 1.0,
        }
        return torch.from_numpy(x), torch.from_numpy(y), meta

def make_loaders(df, context=3):
    train = CTSliceDataset(df[df.split=="train"], context, True)
    val   = CTSliceDataset(df[df.split=="val"], context, False)
    test  = CTSliceDataset(df[df.split=="test"], context, False)
    return (
        DataLoader(train, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True),
        DataLoader(val, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True),
        DataLoader(test, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    )


In [ ]:

def dice_np(p, g, eps=1e-7):
    p=p.astype(bool); g=g.astype(bool)
    if p.sum()+g.sum()==0: return 1.0
    return (2*(p&g).sum()+eps)/(p.sum()+g.sum()+eps)

def iou_np(p,g,eps=1e-7):
    p=p.astype(bool); g=g.astype(bool)
    u=(p|g).sum()
    return 1.0 if u==0 else ((p&g).sum()+eps)/(u+eps)

def surface_distances(a,b,spacing=(1.,1.)):
    a=a.astype(bool); b=b.astype(bool)
    if not a.any() and not b.any(): return np.array([0.])
    if not a.any() or not b.any(): return np.array([np.inf])
    a_s = a ^ binary_erosion(a)
    b_s = b ^ binary_erosion(b)
    dt_b = distance_transform_edt(~b_s, sampling=spacing)
    dt_a = distance_transform_edt(~a_s, sampling=spacing)
    return np.concatenate([dt_b[a_s], dt_a[b_s]])

def hd95_np(p,g,spacing=(1.,1.)):
    d=surface_distances(p,g,spacing)
    return float(np.percentile(d,95)) if np.isfinite(d).all() else np.inf

def assd_np(p,g,spacing=(1.,1.)):
    d=surface_distances(p,g,spacing)
    return float(d.mean()) if np.isfinite(d).all() else np.inf

def surface_dice_np(p,g,tolerance_mm=2.0,spacing=(1.,1.)):
    p=p.astype(bool); g=g.astype(bool)
    if not p.any() and not g.any(): return 1.0
    if not p.any() or not g.any(): return 0.0
    ps=p ^ binary_erosion(p); gs=g ^ binary_erosion(g)
    dtg=distance_transform_edt(~gs,sampling=spacing)
    dtp=distance_transform_edt(~ps,sampling=spacing)
    good = (dtg[ps] <= tolerance_mm).sum() + (dtp[gs] <= tolerance_mm).sum()
    den = ps.sum()+gs.sum()
    return float(good/(den+1e-7))


In [ ]:

def soft_dice_loss(logits, target, eps=1e-6):
    p=torch.sigmoid(logits)
    inter=(p*target).sum((1,2,3))
    den=p.sum((1,2,3))+target.sum((1,2,3))
    return (1-(2*inter+eps)/(den+eps)).mean()

def focal_loss(logits,target,alpha=.25,gamma=2.0):
    bce=F.binary_cross_entropy_with_logits(logits,target,reduction="none")
    p=torch.sigmoid(logits)
    pt=p*target+(1-p)*(1-target)
    at=alpha*target+(1-alpha)*(1-target)
    return (at*((1-pt)**gamma)*bce).mean()

def sobel_edges(x):
    kx=torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]],dtype=x.dtype,device=x.device).view(1,1,3,3)
    ky=torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]],dtype=x.dtype,device=x.device).view(1,1,3,3)
    gx=F.conv2d(x,kx,padding=1); gy=F.conv2d(x,ky,padding=1)
    return torch.sqrt(gx*gx+gy*gy+1e-6)

def boundary_loss(logits,target):
    return F.l1_loss(sobel_edges(torch.sigmoid(logits)), sobel_edges(target))

def total_loss(logits,target,loss_cfg):
    val=0.
    if loss_cfg.get("focal",False): val += focal_loss(logits,target)
    if loss_cfg.get("dice",False): val += soft_dice_loss(logits,target)
    if loss_cfg.get("boundary",False): val += boundary_loss(logits,target)
    return val

@torch.no_grad()
def quick_val_dice(model, loader):
    model.eval(); vals=[]
    for x,y,_ in loader:
        x=x.to(DEVICE); y=y.to(DEVICE)
        p=(torch.sigmoid(model(x))>=THRESHOLD).float()
        inter=(p*y).sum((1,2,3)); den=p.sum((1,2,3))+y.sum((1,2,3))
        vals.extend(((2*inter+1e-7)/(den+1e-7)).cpu().numpy().tolist())
    return float(np.mean(vals)) if vals else np.nan

def train_model(model, train_loader, val_loader, loss_cfg, seed, tag):
    seed_everything(seed)
    model=model.to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(EPOCHS,1))
    best=-1; best_state=None
    epochs=EPOCHS if FULL_TRAINING else 2
    for ep in range(epochs):
        model.train()
        for x,y,_ in train_loader:
            x=x.to(DEVICE); y=y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits=model(x)
            loss=total_loss(logits,y,loss_cfg)
            loss.backward(); opt.step()
        sch.step()
        vd=quick_val_dice(model,val_loader)
        if vd>best:
            best=vd
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        if ep==0 or (ep+1)%10==0 or ep==epochs-1:
            print(tag, "epoch", ep+1, "val Dice", round(vd,4))
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def evaluate(model, loader, tag="model"):
    model.eval(); rows=[]
    for x,y,meta in loader:
        x=x.to(DEVICE)
        probs=torch.sigmoid(model(x)).cpu().numpy()[:,0]
        gt=y.numpy()[:,0]
        for j in range(len(probs)):
            pred=probs[j]>=THRESHOLD; g=gt[j]>=.5
            sx=float(meta["spacing_x"][j]); sy=float(meta["spacing_y"][j])
            rows.append({
                "model":tag,
                "patient_id":meta["patient_id"][j],
                "slice_index":int(meta["slice_index"][j]),
                "dice":dice_np(pred,g), "iou":iou_np(pred,g),
                "hd95":hd95_np(pred,g,(sy,sx)), "assd":assd_np(pred,g,(sy,sx)),
                "surface_dice_2mm":surface_dice_np(pred,g,2.0,(sy,sx))
            })
    return pd.DataFrame(rows)

def summarize(df):
    return df.groupby("model")[["dice","iou","hd95","assd","surface_dice_2mm"]].agg(["mean","std"])


In [ ]:

# ========================= MODEL FACTORY =========================
# REPLACE THIS CELL with your actual implementation/import.
#
# Required:
#   build_model(variant) -> torch.nn.Module
#
# variant may contain:
#   name, in_channels, use_se2, use_boundary, context
#
# Example:
# from my_models import CTSE2, ModSE2, ConventionalUNet
# def build_model(v):
#     if v["name"] == "CT-SE2":
#         return CTSE2(in_channels=v["in_channels"], use_se2=v.get("use_se2",True))
#     if v["name"] == "Mod-SE2":
#         return ModSE2(in_channels=v["in_channels"])
#     return ConventionalUNet(in_channels=v["in_channels"])

def build_model(variant):
    raise NotImplementedError(
        "Paste/import your CT-SE(2), prior Mod-SE(2), and conventional-control model "
        "in the MODEL FACTORY cell, then Run All."
    )


## Experiment
Loss-function ablation with boundary-specific metrics. This directly tests whether boundary supervision improves contour localization.

In [ ]:

df=pd.read_csv(MANIFEST_CSV)
loss_variants=[
 ("Dice only",{"focal":False,"dice":True,"boundary":False}),
 ("Focal + Dice",{"focal":True,"dice":True,"boundary":False}),
 ("Dice + Boundary",{"focal":False,"dice":True,"boundary":True}),
 ("Focal + Dice + Boundary",{"focal":True,"dice":True,"boundary":True}),
]


In [ ]:

out=[]
for label,lc in loss_variants:
    v={"name":"CT-SE2","context":3,"in_channels":3,"use_se2":True,"loss_cfg":lc}
    tr,va,te=make_loaders(df,3)
    for seed in SEEDS:
        print("\n###",label,"seed",seed)
        m=build_model(v)
        m=train_model(m,tr,va,lc,seed,label)
        rr=evaluate(m,te,label)
        rr["seed"]=seed; rr["loss"]=label
        out.append(rr)
res=pd.concat(out,ignore_index=True)
res.to_csv(os.path.join(OUTPUT_DIR,"04_boundary_loss_ablation.csv"),index=False)
patient=res.groupby(["loss","seed","patient_id"],as_index=False)[["dice","iou","hd95","assd","surface_dice_2mm"]].mean()
summary=patient.groupby("loss")[["dice","iou","hd95","assd","surface_dice_2mm"]].agg(["mean","std"])
display(summary)
summary.to_csv(os.path.join(OUTPUT_DIR,"04_boundary_loss_summary.csv"))


In [ ]:

# Publication-oriented plots
for metric in ["dice","hd95","assd","surface_dice_2mm"]:
    means=patient.groupby("loss")[metric].mean()
    errs=patient.groupby("loss")[metric].std()
    fig,ax=plt.subplots(figsize=(8,4))
    ax.bar(range(len(means)),means.values,yerr=errs.values,capsize=4)
    ax.set_xticks(range(len(means))); ax.set_xticklabels(means.index,rotation=20,ha="right")
    ax.set_ylabel(metric); ax.set_title("Boundary-loss ablation: "+metric)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR,f"04_{metric}.png"),dpi=200)
    plt.show()
